<img src=../figures/Brown_logo.svg width=50%>

## Data-Driven Design & Analyses of Structures & Materials (3dasm)

## Lecture 21


### Martin van der Schelling | <a href = "mailto: m.p.vanderschelling@tudelft.nl">m.p.vanderschelling@tudelft.nl</a>  | Doctoral Candidate

### Miguel A. Bessa | <a href = "mailto: miguel_bessa@brown.edu">miguel_bessa@brown.edu</a>  | Associate Professor

### Elvis Aguero | <a href = "mailto: elvis_vera@brown.edu">elvis_vera@brown.edu</a>  | PhD candidate

## Outline of today

Provide an overview of your Final Project for the course.

This final project is inspired by the following research article:

* **Bessa, M. A., Glowacki, P., & Houlder, M. (2019). Bayesian Machine Learning in Metamaterial Design: Fragile Becomes Supercompressible. Advanced Materials, 31(48), 1–6. https://doi.org/10.1002/adma.201904845**

It focused on transforming a fragile polymer material used in 3D printing into a supercompressible metamaterial by designing its geometry.

The data was generated by Finite Element Analyses (using [ABAQUS](https://www.3ds.com/products/simulia/abaqus) together with `f3dasm`).

However, you will not need to generate the data using ABAQUS. Instead, you will be provided with a dataset. This way, you can focus on the Machine Learning and Optimization tasks and avoid running time-consuming simulations.

Still, today we simulate **one design** ourselves, with ABAQUS on Oscar (Brown's computing cluster), to see how each row of the datasets was made.

## Designing supercompressible metamaterials

<img src=../figures/supercompressible_process.png width=65%, align='left'>

<img src=../figures/supercompressible_schematic.png width=30%, align='right'>

<!--- <br><br><br><br><br><br><br><br><br><br><br><br><br><br><br> --->

* Parametrizing your structure from a concept
* Sampling designs
* Run finite element simulation (ABAQUS)
* Learn the objective for the full range of designs
* Manufacture your found optimum

## Datasets

The supercompressible meta-material is parameterized by **7 geometric parameters and 2 material parameters**:
* The geometry is defined by the top and bottom diameters, $D_1$ and $D_2$, 
* the height $P$
* the cross-section parameters of the vertical longerons: area $A$, 
* moments of inertia $I_x$ and $I_y$, 
* torsional constant $J$.
* The isotropic material is defined by its elastic constants: Young's modulus $E$ and shear modulus $G$.

$$
\frac{D_1-D_2}{D_1},\ \frac{P}{D_1},\ \frac{I_x}{D_1^4},\ \frac{I_y}{D_1^4},\ \frac{J}{D_1^4},\ \frac{A}{D_1^2}, \frac{G}{E}
$$

| expression |	parameter name |
| ----------- | --------------- |
| $\frac{D_1-D_2}{D_1}$ |	`ratio_top_diameter`
|$\frac{P}{D_1}$|	`ratio_pitch`
|$\frac{I_x}{D_1^4}$|	`ratio_Ixx`
|$\frac{I_y}{D_1^4}$|	`ratio_Iyy`
|$\frac{J}{D_1^4}$|	`ratio_J`
|$\frac{A}{D_1^2}$|	`ratio_area`
|$\frac{G}{E}$|	`ratio_shear_modulus`



This is a **7-dimensional problem** and learning the response surface may require a **significant amount of training points**

Therefore, you will also consider a **simpler version** of the problem in 3 dimensions, defined by constraining the longerons' cross-section to be circular with diameter $d$, and choosing a particular material, leading to the following 3 features:

$$
\frac{d}{D_1}, \frac{D_1-D_2}{D_1},\ \frac{P}{D_1}
$$

| expression |	parameter name |
| ----------- | --------------- |
$\frac{D_1-D_2}{D_1}$|	`ratio_top_diameter`
$\frac{P}{D_1}$	|`ratio_pitch`
$\frac{d}{D_1}$	|`ratio_d`

In addition to the 7 input variables (or 3 in the simplified case), each design of the supercompressible metamaterial needs to be **perturbed by a small geometric imperfection**.

- Why?

The metamaterial becomes supercompressible because the vertical longerons **buckle**, i.e. undergo a mechanical instability and dramatically coil.

<center><img src=../figures/supercompressible_process.png width=65%></center>

ABAQUS provides a simple way to induce a geometric instability:

1. **Linear buckling analysis** of a metamaterial design considering perfect geometry:
    - Each simulation outputs the buckling load for each mode ("**loads**" output variable), and the maximum displacement of each mode ("**max_disps**" output variable) that are needed to scale the amplitude of the applied geometric imperfections (see below).
    

2. **Postprocess linear buckling analysis** to determine the buckling modes of that design:
    - Use the buckling modes and apply an amplitude value for each of those modes, and add them to create a perturbed geometry;
    - Here we use only the first buckling mode to seed geometric imperfections for each metamaterial design;
    - We only care about a design if its first buckling mode leads to torsion of the top ring (indication of supercompressibility). In that case, we update the "**coilable**" output variable to "coilable = 1". Otherwise, "coilable = 0" and the subsequent postbuckling response prediction of that design is **not** conducted.

Once the linear buckling analysis finishes, and **if that design is supercompressible** ("coilable = 1"), then we continue as follows:

3. **Postbuckling analysis** (RIKS) of metamaterials with "coilable = 1":
    - Use the first buckling mode from the linear buckling analysis to create an imperfect geometry scaled so that its largest displacement is a random amplitude $a$ (in mm), with $\ln a \sim \mathcal{N}(-2.705, 0.294^2)$. This creates some aleatoric uncertainty in the data, and makes it possible to predict the postbuckling response by the RIKS method.

4. **Postprocess postbuckling analysis**:
    - Collect 4 outputs for that design: reaction moments "**RM**", reaction forces "**RF**" at the bottom ring; displacements "**U**" and rotations "**UR**" at the top ring (where load is applied).
    - Check the largest strain in the longerons up to full compression. If it exceeds 2%, update to "coilable = 2", because such a design is supercompressible but yields.
    
Done: this completes one simulation of a particular metamaterial design.

**NOTE**: In the project you will not have to use ABAQUS because you will be provided with the dataset already. I am giving you this information just so you understand how the data was generated, and today you will run one design. This course is not a Finite Element course 😉

## The usual way to create this dataset

<img src="../figures/blocks_supercompressible_original.png" width="50%">

Some problems may arise:
* How do we **parametrize** the design efficiently?
* How do we run this in **parallel** or on a **high performance computing cluster**?
* How do we **avoid reinventing the wheel** (e.g. calling Abaqus with python scripts)

## You guessed it! With f3dasm!

<img src="../figures/supercompressible_colored.png" width="100%">

- For the 7D case, a dataset with $1000$ experiments that is parametrized by 7 parameters (`supercompressible_7d`) will be provided.
- For the 3D case, a dataset with $1000$ experiments (`supercompressible_3d`) will be provided.


Let's recreate the 3D dataset. First create the  parametrized domain:

In [1]:
from f3dasm.design import Domain

We add the parameters and their respective bounds:

In [2]:
domain = Domain()

domain.add_float('ratio_top_diameter', low=0.0, high=0.8)
domain.add_float('ratio_pitch', low=0.25, high=1.5)
domain.add_float('ratio_d', low=0.004, high=0.073)
domain.add_float(name='imperfection', low=0.0, high=float('inf')) # lognormal: any positive value

The Design of Experiments module is very simple:
- [Sobol sequence sampling](https://salib.readthedocs.io/en/latest/_modules/SALib/sample/sobol_sequence.html) is used for the remaining parameters `'ratio_top_diameter'`, `'ratio_pitch'` and `'ratio_d'`

Note: The amplitude of geometric imperfections is sampled from the previously mentioned lognormal distribution for each design. It is **not a design variable**: it acts as a hidden rv, like the reaction time $z$ in the car stopping distance problem.

We still add it to a `Domain`, but only for **bookkeeping**: each experiment has to record the imperfection that its simulation used. The datasets you will receive only contain the three design variables as inputs.

Still, in  order for us to sample both types of variables, we can split the domain in two subdomains and sample from them individually:

In [3]:
domain_imperfection = Domain()
domain_imperfection.add_float(name='imperfection', low=0.0, high=float('inf')) # lognormal: any positive value

domain_rest = Domain()
domain_rest.add_float('ratio_top_diameter', low=0.0, high=0.8)
domain_rest.add_float('ratio_pitch', low=0.25, high=1.5)
domain_rest.add_float('ratio_d', low=0.004, high=0.073)

We create two separate `f3dasm.ExperimentData` objects and later join them together!

In [4]:
from f3dasm import ExperimentData, create_sampler

experimentdata_imperfection = ExperimentData(domain=domain_imperfection)
experimentdata_rest = ExperimentData(domain=domain_rest)

An implementation of sobol sequence sampling is available as a built-in default, so we can create that sampler right away and `call` it on our `ExperimentData`:

In [5]:
N = 1 # One design today (each one takes about a minute on Oscar). The datasets have 1000!

sobol_sampler = create_sampler('sobol_sampler', seed=123)

experimentdata_rest = sobol_sampler.call(experimentdata_rest, n_samples=N)
experimentdata_rest

jobs   input                               
        ratio_d ratio_pitch ratio_top_diameter
0  open   0.004        0.25                0.0

For the lognormal distribution, we have to provide our own sampler. A sampler in `f3dasm` is a `Block`: a class with a `call` method that receives the `ExperimentData`, adds new samples to it and returns it.
- The number of samples is passed to `call` as `n_samples`
- Everything specific to *this* sampler goes into `__init__`: the mean and standard deviation of the lognormal distribution, and a seed for reproducibility

In [6]:
import numpy as np
from f3dasm import Block


class LogNormalSampler(Block):
    """Sampler that draws a single parameter from a lognormal distribution."""

    def __init__(self, mean: float, sigma: float, seed: int = None):
        self.mean = mean
        self.sigma = sigma
        self.seed = seed

    def call(self, data: ExperimentData, n_samples: int, **kwargs) -> ExperimentData:
        rng = np.random.default_rng(self.seed) # Create a numpy generator with a random seed
        samples = rng.lognormal(mean=self.mean, sigma=self.sigma, size=n_samples)

        # Wrap the samples in a new ExperimentData with the same domain ...
        new_data = ExperimentData(input_data=samples, domain=data.domain,
                                  project_dir=data.project_dir)

        # ... and append them to the data we were given
        return data + new_data

Now we can sample using this custom sampler:

In [7]:
log_normal_sampler = LogNormalSampler(mean=-2.705021452041446,
                                      sigma=0.293560379208524,
                                      seed=123)

experimentdata_imperfection = log_normal_sampler.call(experimentdata_imperfection, n_samples=N)
experimentdata_imperfection

,jobs,input
,,imperfection
0,open,0.050017


Lastly, we join both `f3dasm.ExperimentData` objects together using the `join()` method:

In [8]:
experimentdata = experimentdata_rest.join(experimentdata_imperfection)
experimentdata

jobs        input                                       
        imperfection ratio_d ratio_pitch ratio_top_diameter
0  open     0.050017   0.004        0.25                0.0

Now we are ready to evaluate our samples with a datagenerator:
The simulation consists of two ABAQUS simulations (one after the other):

* The first one is a linear buckling simulation
* The second one is a RIKS simulation

We split up the components of the simulation in three parts:

* **pre-processing**: combine the design and the other parameters/constants to an input file
* **processing**: run the simulation
* **post-processing**: convert the output of the simulator back to `f3dasm`

<img src="../figures/blocks_supercompressible_datagenerator.png" width="100%">

## Data generation: getting the output data

* As mentioned, we want to use **non-linear finite element analyses** to predict the **complete buckling and post-buckling behavior**. 
* From the analyses, we can understand if a **material is coilable**, compute the **critical buckling stress** $\sigma_{crit}$ (critical buckling load of the linear buckling analysis divided by the area of the bottom ring) and the elastic **energy absorbed** $E_{abs}$ (integral of the effective stress-strain response from the postbuckling simulation).
* Due to unsuccessful simulations, there are **missing points in the datasets**. This is very common in real datasets (simulations sometimes can have convergence issues, etc.).

## Running ABAQUS on Oscar

ABAQUS is installed on **Oscar**, Brown's computing cluster. The file `oscar_helper.py`, next to this notebook, gives you an `f3dasm` `DataGenerator` called `OscarAbaqus`. For each open experiment it:

1. sends the design and the four ABAQUS scripts of the folder `scripts/` to Oscar, as one **Slurm job** (Slurm is the program that shares the computers of a cluster),
2. waits for the job to finish (about a minute, plus the time the job waits in the queue),
3. copies the results back and stores `coilable`, `sigma_crit` and `energy` in the `ExperimentData`.

**Note: connecting to Oscar.** You need an Oscar account, the Brown VPN when you are off campus, and an entry `oscar-campus` in the file `~/.ssh/config` of your computer:

```
Host oscar-campus
    HostName sshcampus.ccv.brown.edu
    User <your Brown username>
    ControlMaster auto
    ControlPath ~/.ssh/cm-%r@%h:%p
    ControlPersist 8h
```

Open a terminal and run `ssh oscar-campus` once (password and Duo). The `ControlMaster` lines keep that connection open for 8 hours, and the notebook uses it without asking again. An ssh key registered with CCV works too. If the notebook cannot connect, `OscarAbaqus` stops with a message that says so.

**Note: what the job does.** It first checks that it runs on the compute node that Slurm gave it (never on a login node, which everybody shares). Then it loads the ABAQUS module and runs the four scripts: linear buckling and its post-processing, then, if the design is coilable, Riks and its post-processing. It writes small JSON files, and deletes the large ABAQUS files. The labels and the energy are computed in this notebook, with the rules of the datasets of the Final Project (`outputs` in `oscar_helper.py`). Each job has its own folder in `~/scratch/3dasm_lecture21` on Oscar, where you can read the ABAQUS messages if something fails.

In [9]:
from oscar_helper import OscarAbaqus

simulator = OscarAbaqus()  # connects with the ssh alias 'oscar-campus'

The linear buckling simulation requires some additional parameters that are **not** part of the Design of Experiments because they do not matter (we used non-dimensional inputs). Still, the simulations need them, and `OscarAbaqus` passes them to the scripts:

* young_modulus = 3500.0
* n_longerons = 3
* bottom_diameter = 100.0
* ratio_shear_modulus = 0.3677
* circular = True

**Note**: If curious, look at the script `scripts/supercompressible_lin_buckle.py` to understand how the simulation is created.

Anyway, recall the `experimentdata` object after we did the Design of Experiments (one design):

In [10]:
experimentdata

jobs        input                                       
        imperfection ratio_d ratio_pitch ratio_top_diameter
0  open     0.050017   0.004        0.25                0.0

And now we simply evaluate the `f3dasm.ExperimentData` by running both ABAQUS simulations on Oscar:

In [11]:
experimentdata = simulator.call(experimentdata)

Submitted Slurm job 7824125 (l21_20261022T214318_86483) for {'ratio_d': 0.004, 'ratio_pitch': 0.25, 'ratio_top_diameter': 0.0, 'imperfection': 0.05001720209950288}


      2 s  PENDING


     27 s  RUNNING


     90 s  COMPLETED


  reversibly coilable (largest strain 0.43%): coilable 1.0, sigma_crit 0.0004942 kPa, energy 0.0003069 kJ/m^3


Once this finishes, we can see the `experimentdata` object again to verify that the outputs were obtained:

In [12]:
experimentdata

jobs        input                                          output  \
            imperfection ratio_d ratio_pitch ratio_top_diameter coilable   
0  finished     0.050017   0.004        0.25                0.0      1.0   

                        
     energy sigma_crit  
0  0.000307   0.000494

This design is the first row of `supercompressible_3d`, where $\sigma_{crit} = 0.000494$ kPa and $E_{abs} = 0.00029$ kJ/m$^3$, simulated with an amplitude $a = 0.083$ mm. Our run used $a = 0.050$ mm. It gives the same $\sigma_{crit}$ and $E_{abs} = 0.00031$ kJ/m$^3$.
- $\sigma_{crit}$ comes from the linear buckling analysis of the perfect geometry, so it does not depend on $a$.
- $E_{abs}$ comes from the Riks analysis of the imperfect geometry, so it changes with $a$, which makes it a random variable.

Some simulations will lead to errors, and some will simply not run because the design was determined as not supercompressible ("coilable = 0") and we do not care for those postbuckling responses.

As you can see, this `DataGenerator` is more complicated than a simple "analytical function" like the Car Stopping Distance problem, or the benchmark functions that are included in `f3dasm`. Yet, the data-driven process follows the same logic, independently of how advanced each of its modules is.

## Your Final Project

Your Final Project starts now! Now that you know how the two datasets for the supercompressible metamaterial were created, you can put your Machine Learning and Optimization knowledge into practice...

* The **coilable** output can be used to train **classification models**
* The **critical buckling stress** and elastic **energy absorbed** can be used to train **regression models**
* **Optimization methods** can find the best designs (after you did classification and regression).

<img src=../figures/supercompressible_regression_classification.png width=50%>

You have all the tools you need to start the project.

The remaining lectures in the course will provide you more knowledge by covering, among other things:

* Classification machine learning models

* Optimization methods (introduction)

* Artificial neural network models

But you are strongly encouraged to train and test **new machine learning** models and **optimization** methods!

You do not need to be confined to the methods we cover in the course because you have the basic theory to do whatever you want!

## Field guide: what this lecture adds

### Good practices

**1. Frame the problem**
- **Send the simulation, keep the record**: an expensive simulation can run on a cluster behind the same `DataGenerator` interface, while the `ExperimentData` stays in your notebook.

The whole [field guide](https://github.com/bessagroup/3dasm_course/blob/main/FIELD_GUIDE.md) collects the key concepts and good practices of every lecture.

### See you next class

Have fun!